### Importation des librairies

In [38]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
import pandas as pd
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
import warnings
from sklearn.pipeline import make_pipeline

### Importation des fichiers

In [39]:
x_train = pd.read_csv('data/x_train.csv', index_col=0)
x_test = pd.read_csv('data/x_test.csv', index_col=0)
y_train = pd.read_csv('data/y_train.csv', index_col=0)
sample_submission = pd.read_csv('data/new_output_sample.csv', index_col=0)

print("Dimensions x_train :", x_train.shape)
print("Dimensions y_train :", y_train.shape)
print("Dimensions x_test :", x_test.shape)
print("Dimensions sample_submission :", sample_submission.shape)

Dimensions x_train : (1057, 21000)
Dimensions y_train : (1057, 1000)
Dimensions x_test : (1057, 38140)
Dimensions sample_submission : (1057, 1000)


### Fonctions utiles

In [40]:
# Génération du fichier de soumission
def generer_soumission(soumission, nom_sortie):
    is_valid = (soumission.shape == sample_submission.shape) # Vérification du format avec le format cible
    if is_valid:
        soumission.to_csv(f'{nom_sortie}.csv') # Génération du fichier de soumission selon le nom entré en paramètre
        print(f"Fichier '{nom_sortie}.csv' généré avec succès !") 
    else:
        print("Attention, les dimensions ne correspondent pas au fichier sample.")

In [41]:
def benchmark(column):
    col = column.copy()
    col = col.interpolate(method='linear', limit_direction='both')
    return col

In [ ]:
def echantilloner ():
    holed_cols = [col for col in x_test.columns if 'holed' in col]
    complete_cols = [col for col in x_test.columns if 'holed' not in col]
    x_test_filled = x_test[holed_cols].copy()
    X_features_reduced = x_test[complete_cols].sample(n=2000, axis=1, random_state=67) # Permet de faire un échantillon parmis toutes les données pour accélérer le calcul (le calcul de base prend environ 1h sans échantillon)
    return holed_cols, x_test_filled, X_features_reduced

### Soumission de base (interpolation linéaire)

Fonction d'interpolation linéaire (remplissage des trous par une ligne droite, comme lors des précédents TP, pour éviter une erreur nan)

In [43]:
def interpolation_lineaire(column):
    return column.interpolate(method='linear', limit_direction='both') # Evite les erreurs nan comme lors des derniers TP

In [44]:
holed_cols_test = [col for col in x_test.columns if 'holed' in col]
y_pred_test = x_test[holed_cols_test].apply(interpolation_lineaire, axis=0)
submission = y_pred_test.loc[sample_submission.index, sample_submission.columns]

generer_soumission(submission, "interpolation_lineaire")

Fichier 'interpolation_lineaire.csv' généré avec succès !


### Régression linéaire
Le score donné par cette regression linéaire est de 103,18550320391749 

In [45]:
def regression_lineaire():

    holed_cols, complete_cols, x_test_filled, X_features_reduced = echantilloner()
    print("Lancement de la régression linéaire avec recherche optimisée...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        # Enlève les messages d'erreur inutiles
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        top_vars = corrs.sort_values(ascending=False).head(10).index
        
        model = LinearRegression()
        model.fit(X_features_reduced.loc[mask_known, top_vars], target_series[mask_known])        
        predictions = model.predict(X_features_reduced.loc[mask_missing, top_vars])
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "regression_lineaire")
    return submission

regression_lineaire()



Lancement de la régression linéaire avec recherche optimisée...


100%|██████████| 1000/1000 [03:40<00:00,  4.54it/s]


Fichier 'regression_lineaire.csv' généré avec succès !


,holed_1,holed_2,holed_3,holed_4,holed_5,holed_6,holed_7,holed_8,holed_9,holed_10,...,holed_991,holed_992,holed_993,holed_994,holed_995,holed_996,holed_997,holed_998,holed_999,holed_1000
Horodate,,,,,,,,,,,,,,,,,,,,,
2023-01-09 00:00:00,1061.000000,118.994488,148.250917,20.000000,123.688939,148.0,70.000000,183.000000,88.0,25.599871,...,38.0,168.000000,32.0,355.726451,445.0,161.984951,23.910676,767.0,2029.0,129.000000
2023-01-09 00:30:00,1041.000000,107.942475,99.000000,42.000000,138.000000,164.0,59.000000,170.000000,83.0,21.617781,...,27.0,82.000000,48.0,365.643981,492.0,207.000000,22.777282,773.0,1698.0,113.000000
2023-01-09 01:00:00,995.000000,104.979209,105.000000,19.000000,145.000000,93.0,119.000000,403.000000,60.0,19.433466,...,37.0,102.000000,44.0,401.095452,461.0,218.000000,33.021988,613.0,1737.0,123.309864
2023-01-09 01:30:00,998.000000,111.771847,129.384671,34.000000,270.000000,126.0,505.000000,489.000000,66.0,16.034775,...,13.0,78.000000,33.0,409.459387,491.0,104.000000,-11.390579,691.0,994.0,135.000000
2023-01-09 02:00:00,1040.639313,76.694292,107.000000,21.000000,309.000000,1279.0,395.000000,288.000000,46.0,17.178090,...,62.0,107.000000,36.0,408.031906,451.0,97.000000,32.260896,844.0,1044.0,123.784539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-01-30 22:00:00,1724.000000,97.000000,48.000000,46.030635,424.000000,632.0,125.000000,288.816737,227.0,34.000000,...,146.0,323.129544,137.0,669.000000,881.0,596.000000,147.000000,502.0,1599.0,71.091784
2023-01-30 22:30:00,1677.000000,63.000000,40.000000,31.525330,435.000000,610.0,137.037190,265.963790,259.0,26.000000,...,66.0,317.418140,155.0,909.000000,985.0,681.000000,88.000000,741.0,1887.0,78.000000
2023-01-30 23:00:00,1648.000000,60.000000,72.766140,20.948834,428.000000,535.0,124.864617,165.000000,286.0,39.582017,...,44.0,246.904548,131.0,973.000000,940.0,206.000000,65.000000,809.0,1882.0,57.000000
